# Quick Demo - Brain Tumor Detection

**Purpose**: Simple demonstration of the trained model for recruiters/reviewers.

This notebook shows how to:
1. Load the trained YOLOv11s model
2. Make predictions on brain MRI images
3. Visualize detection results

## Setup

In [ ]:
import sys
sys.path.append('../src')

from utils.model import load_model, predict
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Libraries imported successfully!")

## Step 1: Load Trained Model

Load the best performing model from training.

In [ ]:
# Load the trained PyTorch model
model_path = '../models/train/weights/best.pt'

print(f"Loading model from: {model_path}")
model = load_model(model_path)
print("✅ Model loaded successfully!")
print(f"Model type: {type(model)}")
print(f"Classes: {model.names}")

## Step 2: Load Sample Images

Load validation batch images from MLflow artifacts to demonstrate predictions.

In [ ]:
# Display some validation predictions from training
mlflow_artifacts = Path('../mlruns/1/4be4c4c61f414f698c78cf88a6a18673/artifacts')

if mlflow_artifacts.exists():
    # Show validation batch predictions
    val_pred_path = mlflow_artifacts / 'val_batch0_pred.jpg'
    
    if val_pred_path.exists():
        fig, ax = plt.subplots(figsize=(16, 12))
        pred_img = Image.open(val_pred_path)
        ax.imshow(pred_img)
        ax.set_title('Model Predictions on Validation Batch', fontsize=16, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        
    print("✅ Sample predictions displayed!")
else:
    print("⚠ MLflow artifacts not found, skipping visualization")

## Step 3: Make Predictions on New Images

Function to make predictions and visualize results.

In [ ]:
def visualize_prediction(model, image_path, conf_threshold=0.25):
    """
    Make prediction and visualize results.
    
    Args:
        model: Loaded YOLO model
        image_path: Path to image file
        conf_threshold: Confidence threshold for detections
    """
    # Read image
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Make prediction
    results = predict(model, img_rgb, conf_thres=conf_threshold)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Original image
    axes[0].imshow(img_rgb)
    axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Prediction
    if len(results) > 0 and hasattr(results[0], 'plot'):
        result_img = results[0].plot()
        axes[1].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
        axes[1].set_title('Detection Results', fontsize=14, fontweight='bold')
    else:
        axes[1].imshow(img_rgb)
        axes[1].set_title('No Detections', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print detections
    if len(results) > 0 and hasattr(results[0], 'boxes') and len(results[0].boxes) > 0:
        print(f"\n🎯 Found {len(results[0].boxes)} detection(s):")
        for i, box in enumerate(results[0].boxes):
            class_id = int(box.cls[0])
            conf = float(box.conf[0])
            class_name = model.names[class_id]
            print(f"   {i+1}. {class_name}: {conf:.2%} confidence")
    else:
        print("\n❌ No detections found")

print("✅ Visualization function defined")

## Step 4: Try it on Your Own Images!

To test with your own images:
1. Place image in a folder (e.g., `test_images/`)
2. Update the path below
3. Run the cell

In [ ]:
# Example: Visualize prediction on a test image
# UNCOMMENT and UPDATE the path to your test image

# test_image_path = Path('path/to/your/test/image.jpg')
# if test_image_path.exists():
#     visualize_prediction(model, test_image_path, conf_threshold=0.25)
# else:
#     print(f"Image not found: {test_image_path}")

print("💡 Update the path above to test with your own images!")

## Model Performance Summary

### Key Metrics:
- **Model**: YOLOv11s
- **mAP@0.5**: ~92%
- **Inference Speed**: ~83 FPS (PyTorch), ~120 FPS (ONNX)
- **Model Size**: 22 MB (PyTorch), 11 MB (ONNX)

### Classes:
1. **Glioma**: Brain tumor type
2. **Meningioma**: Tumor in brain membranes
3. **Pituitary**: Pituitary gland tumor
4. **No Tumor**: Normal brain scan

## Next Steps

For more detailed analysis, see:
- `01_EDA_Brain_Tumor_Dataset.ipynb` - Data exploration
- `02_Model_Experiments_Comparison.ipynb` - Model selection process

To use the model in production:
- Run `streamlit run src/servering/app.py` for web interface
- Use `src/servering/api.py` for REST API